# LG-02: Tools 深度掌握

> **阶段**: LG-02 | **难度**: 初级 | **预计时长**: 4-5 小时 | **依赖**: LG-01

## 学习目标
- 掌握 5 种 Tool 定义方法，按场景选型
- 理解 `return_direct` 模式控制结果流向
- 掌握生产级 Tool 五层设计模式
- 了解 Tool 缓存机制与上下文管理策略
- 掌握 Context 注入（InjectedState, InjectedStore）
- 了解 Prompt Templates 基础用法

---

## 开场引入

上节课我们画了一张静态的图——WeatherBot，输入确定、流程确定、输出确定。但真实世界不是这样：用户说『查北京天气』，你要调天气 API；用户说『查订单』，你要调订单系统；用户说『帮我算一下 23 的平方根』，你要调计算器。

今天这节课聚焦一个核心问题：**图怎么调用外部能力？（Tool）**

更重要的是——Tool 不仅仅是大模型的『外挂』，它本身就是 LangGraph 的一等公民。你定义好的 Tool，可以直接当成图的节点来用，定义一次，在任何图里复用。

In [ ]:
# 安装依赖（如未安装请取消注释）
# !pip install -U langgraph langchain langchain-openai pydantic

## 1. Tool 的五种定义方式

LangChain / LangGraph 提供了五种创建 Tool 的方式，从简单原型到生产级运行时注入。

### 方式 1：`@tool` 装饰器（最简单，快速原型）

框架自动从 docstring + 类型注解生成 tool schema。**docstring = LLM 看到的工具说明书**，写不清楚，LLM 就不会用。

In [ ]:
from langchain_core.tools import tool

# ==========================================
# 方式 1: @tool 装饰器
# ==========================================
@tool
def query_weather(city: str) -> str:
    """查询指定城市的实时天气。

    Args:
        city: 城市名称，如"北京"、"Shanghai"

    Returns:
        天气描述字符串，如"北京今天晴，25度"
    """
    weather_db = {
        "北京": "晴，25°C",
        "上海": "多云，28°C",
        "深圳": "小雨，30°C",
    }
    return weather_db.get(city, f"未找到 {city} 的天气数据")

# 查看 tool schema
print(f"Tool 名称: {query_weather.name}")
print(f"Tool 描述: {query_weather.description}")
print(f"参数 schema: {query_weather.args}")
print()

# 测试调用
result = query_weather.invoke({"city": "北京"})
print(f"调用结果: {result}")

### 方式 2：`@tool` + `args_schema`（复杂参数，生产推荐）

当参数多、需要校验、需要详细描述时，用 Pydantic BaseModel。

**为什么用 `args_schema`？**
- 参数有默认值时，LLM 知道哪些是可选的
- `Field(description=...)` 会被直接写入 tool schema
- Pydantic 自动做类型校验，LLM 传错类型会提前报错

In [ ]:
from pydantic import BaseModel, Field

# ==========================================
# 方式 2: @tool + args_schema
# ==========================================
class WeatherInput(BaseModel):
    city: str = Field(description="城市名称，如'北京'、'Shanghai'")
    date: str | None = Field(
        default=None,
        description="查询日期，格式 YYYY-MM-DD。不填则查今天。"
    )
    include_aqi: bool = Field(
        default=False,
        description="是否同时返回空气质量指数"
    )

@tool(args_schema=WeatherInput)
def query_weather_advanced(city: str, date: str | None = None, include_aqi: bool = False) -> str:
    """查询指定城市的实时天气，可选返回空气质量。"""
    result = f"{city}今天晴，25度"
    if include_aqi:
        result += "，空气质量指数 85（良）"
    if date:
        result += f"，查询日期: {date}"
    return result

print(f"Tool 名称: {query_weather_advanced.name}")
print(f"参数 schema:")
for name, info in query_weather_advanced.args.items():
    print(f"  {name}: {info}")
print()

# 测试调用
result = query_weather_advanced.invoke({
    "city": "北京",
    "include_aqi": True
})
print(f"调用结果: {result}")

### 方式 3：`StructuredTool.from_function()`（程序化创建）

适合动态生成 tool、从配置文件中加载 tool 的场景。

In [ ]:
from langchain_core.tools import StructuredTool

# ==========================================
# 方式 3: StructuredTool.from_function
# ==========================================
class CalculatorInput(BaseModel):
    expression: str = Field(description="数学表达式，如 '23 * 45'、'sqrt(16)'")

def calculate_impl(expression: str) -> str:
    try:
        # 安全计算：只允许基本运算
        allowed = {"__builtins__": {}}
        result = eval(expression, allowed, {"__builtins__": {}})
        return str(result)
    except Exception as e:
        return f"计算错误：{e}"

calculator_tool = StructuredTool.from_function(
    func=calculate_impl,
    name="calculator",
    description="计算数学表达式。",
    args_schema=CalculatorInput,
    return_direct=False,       # 默认：结果回到 LLM 继续推理
    handle_tool_error=True,    # 自动捕获异常并返回错误信息
)

print(f"Tool 名称: {calculator_tool.name}")
print(f"return_direct: {calculator_tool.return_direct}")
print(f"handle_tool_error: {calculator_tool.handle_tool_error}")
print()

# 测试调用
result = calculator_tool.invoke({"expression": "23 * 45"})
print(f"23 * 45 = {result}")

result2 = calculator_tool.invoke({"expression": "1 / 0"})
print(f"1 / 0 = {result2}")  # 错误被捕获，不会抛异常

### 方式 4：继承 `BaseTool`（完全自定义，最灵活）

适合需要自定义元数据、权限控制、状态管理的场景。

In [ ]:
from langchain_core.tools import BaseTool
from typing import Type

# ==========================================
# 方式 4: 继承 BaseTool
# ==========================================
class QueryWeatherInput(BaseModel):
    city: str = Field(description="城市名称")

class QueryWeatherTool(BaseTool):
    name: str = "query_weather"
    description: str = "查询指定城市的实时天气。"
    args_schema: Type[BaseModel] = QueryWeatherInput

    def _run(self, city: str) -> str:
        """同步执行"""
        weather_db = {"北京": "晴，25°C", "上海": "多云，28°C"}
        return weather_db.get(city, f"未找到 {city} 的天气数据")

    async def _arun(self, city: str) -> str:
        """异步执行（可选但推荐）"""
        return self._run(city)

weather_tool = QueryWeatherTool()
print(f"Tool 名称: {weather_tool.name}")
print(f"Tool 描述: {weather_tool.description}")

result = weather_tool.invoke({"city": "上海"})
print(f"调用结果: {result}")

### 方式 5：`InjectedState` / `InjectedStore`（LangGraph 运行时注入）

LangGraph 支持通过类型注解将 graph state 或持久化 store 直接注入 tool，**无需 Factory 闭包**即可访问运行时状态。

**为什么用注入机制替代 Factory 闭包？**
- 不需要 `build_xxx_tool()` 工厂函数，tool 本身是无状态的
- state / store 由 LangGraph 运行时自动注入，代码更干净
- 兼容 `create_react_agent`、`ToolNode` 等预构建节点

In [ ]:
from typing import Annotated# InjectedState removed in newer versions; use direct state parameter insteadclass InjectedState: pass  # compat shim, InjectedStore# ==========================================# 方式 5: InjectedState / InjectedStore# ==========================================# 注意: InjectedStore 需要 LangGraph 1.x + 配置好的 store 后端# 这里演示语法，实际运行需要 store 配置@tooldef query_with_context(    city: str,    state: Annotated[dict, InjectedState()],       # 运行时自动注入当前 graph state) -> str:    """查询天气，同时访问 graph state 中的用户信息。"""    user_id = state.get("user_id", "anonymous")    weather_db = {"北京": "晴，25°C", "上海": "多云，28°C"}    weather = weather_db.get(city, "未知")    return f"用户 {user_id} 查询 {city} 天气: {weather}"print(f"Tool 名称: {query_with_context.name}")print(f"参数 schema:")for name, info in query_with_context.args.items():    print(f"  {name}: {info}")print()print("注意: state 参数会被 LangGraph 运行时自动注入，不需要 LLM 传入")

### 五种方式选型总结

| 方式 | 适用场景 | 复杂度 |
|------|---------|--------|
| `@tool` | 参数少、快速验证 | 低 |
| `@tool` + `args_schema` | 参数多、需要校验、生产环境 | 中 |
| `StructuredTool.from_function()` | 动态生成、配置化 | 中 |
| 继承 `BaseTool` | 需要自定义元数据、权限、状态 | 高 |
| `InjectedState` / `InjectedStore` | 预构建 Agent，需访问 state/store | 中 |

## 2. `return_direct`：控制结果流向

`return_direct` 决定 tool 的输出是回到 LLM 继续推理，还是直接返回给用户。

| 场景 | return_direct | 原因 |
|------|--------------|------|
| 查询天气后需要 LLM 整理回复 | False | 原始数据需要 LLM 加工成自然语言 |
| 计算器返回数值 | True | 数值就是最终答案 |
| 查询订单状态后需要判断下一步 | False | 需要根据状态决定后续操作 |
| 返回固定格式的卡片/图片 | True | 前端直接渲染，不需要 LLM 加工 |

In [ ]:
from datetime import datetime

# ==========================================
# return_direct=False（默认）：结果回到 LLM
# ==========================================
@tool
def query_weather_for_llm(city: str) -> str:
    """查询天气。结果需要 LLM 进一步整理回复。"""
    return f"{city}今天晴，25度，东南风2级"

# ==========================================
# return_direct=True：结果直出，不再经过 LLM
# ==========================================
@tool(return_direct=True)
def get_time() -> str:
    """获取当前时间。结果直接展示给用户，不需要 LLM 再加工。"""
    return f"现在是 {datetime.now().strftime('%H:%M')}"

print(f"query_weather_for_llm.return_direct = {query_weather_for_llm.return_direct}")
print(f"get_time.return_direct = {get_time.return_direct}")
print()
print("return_direct=False: 结果回到 LLM，LLM 可以继续思考、调用其他工具")
print("return_direct=True:  结果直出给用户，不再经过 LLM 加工")

## 3. 生产级 Tool 五层设计

```
┌─────────────────────────────────────────────────────────────┐
│                    生产级 Tool 五层设计                       │
├─────────────────────────────────────────────────────────────┤
│ ① 输入层：Pydantic 校验 + 缺失参数检查 + 归一化              │
│ ② 执行层：业务逻辑 + 外部 API 调用                           │
│ ③ 容错层：异常捕获 + 候选匹配 + 重试机制                     │
│ ④ 输出层：LLM 摘要 + 原始数据写入 State                      │
│ ⑤ 透传层：State 写入 + 格式化文本返回 + 日志记录             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ==========================================
# 生产级 Tool 完整示例
# ==========================================

# ① 输入层：Pydantic args_schema
class LibraryAvailabilityInput(BaseModel):
    library_id: str | None = Field(
        default=None, description="书房真实 ID。没有就留空，不要编造。"
    )
    library_name: str | None = Field(
        default=None, description="书房名称，例如 平原轩。"
    )
    booking_date: str | None = Field(
        default=None, description="预约日期，格式 YYYY-MM-DD。没有明确日期时不要猜。"
    )

# 辅助函数：参数归一化
def _normalize_text(value) -> str:
    if value is None:
        return ""
    return str(value).strip()

# ⑤ Factory 模式 + 依赖注入
def build_availability_tool(state: dict, client: dict = None):
    """Factory：根据当前上下文构建 tool 实例。"""

    @tool(
        args_schema=LibraryAvailabilityInput,
        return_direct=False,
    )
    def availability_tool(
        library_id: str | None = None,
        library_name: str | None = None,
        booking_date: str | None = None,
    ) -> str:
        """查询书房可预约时段。"""

        # ① 输入层：缺失参数检查
        missing = []
        if not (library_id or library_name):
            missing.append("书房名称或 ID")
        if not booking_date:
            missing.append("预约日期")
        if missing:
            return f"缺少必要参数：{', '.join(missing)}。请先提供这些信息。"

        # ② 执行层：业务逻辑（模拟）
        try:
            # 模拟查询结果
            result = {
                "library": library_name or library_id,
                "date": booking_date,
                "slots": ["09:00-11:00", "14:00-16:00", "19:00-21:00"]
            }
        except Exception as e:
            # ③ 容错层：异常处理
            return f"查询失败：{str(e)}。请稍后再试。"

        # ③ 容错层：结果校验
        if not result or not result.get("slots"):
            return f"抱歉，{library_name or library_id} 在 {booking_date} 没有可预约时段。"

        # ④ 输出层：格式化摘要
        summary = (
            f"{result['library']} 在 {result['date']} 的可预约时段："
            f"{', '.join(result['slots'])}"
        )

        # ⑤ 透传层：State 写入
        state["availability_result"] = result
        state["booking_candidates"] = result["slots"]

        return summary

    return availability_tool

# 使用
shared_state = {}
availability_tool = build_availability_tool(state=shared_state)

# 测试：缺少参数
result1 = availability_tool.invoke({})
print(f"测试 1 (缺少参数): {result1}")
print()

# 测试：正常查询
result2 = availability_tool.invoke({
    "library_name": "平原轩",
    "booking_date": "2025-06-01"
})
print(f"测试 2 (正常查询): {result2}")
print(f"State 写入结果: {shared_state.get('availability_result')}")

## 4. Tool 缓存机制

### 4.1 LLM 调用缓存（省钱）

相同的 prompt 第二次调用时，直接从缓存返回，不花钱调 API。

In [ ]:
import osfrom langchain_core.globals import set_llm_cachefrom langchain_core.caches import InMemoryCache# ==========================================# LLM 调用缓存# ==========================================# 内存缓存（开发用）set_llm_cache(InMemoryCache())print("LLM 缓存已配置 (InMemoryCache)")print("相同 prompt 第二次调用将直接返回缓存结果")print()print("其他缓存选项:")print("  - SQLiteCache(database_path='.langchain_cache.db')  # 本地持久化")print("  - RedisCache(redis_url=os.getenv('REDIS_URL', 'redis://:redis@localhost:6379/0'))  # 分布式")

### 4.2 Tool 结果缓存

进程内 Tool 缓存推荐用 `functools.lru_cache` 或自定义装饰器。

In [ ]:
from functools import lru_cache
import time

# ==========================================
# Tool 结果缓存
# ==========================================

call_count = 0

@tool
@lru_cache(maxsize=128)
def query_weather_cached(city: str, date: str | None = None) -> str:
    """查询天气（单进程内缓存 128 条）。"""
    global call_count
    call_count += 1
    # 模拟 API 调用延迟
    time.sleep(0.1)
    weather_db = {"北京": "晴，25°C", "上海": "多云，28°C"}
    return weather_db.get(city, f"未找到 {city} 的天气数据")

# 第一次调用（未缓存）
start = time.time()
r1 = query_weather_cached.invoke({"city": "北京"})
t1 = time.time() - start
print(f"第 1 次调用: {r1} (耗时 {t1:.3f}s, 实际调用 API)")

# 第二次调用（命中缓存）
start = time.time()
r2 = query_weather_cached.invoke({"city": "北京"})
t2 = time.time() - start
print(f"第 2 次调用: {r2} (耗时 {t2:.3f}s, 命中缓存)")

print(f"\nAPI 实际调用次数: {call_count}")
print(f"缓存命中率: 50% (2次调用，1次命中)")
print()
print("⚠️ 注意: lru_cache 要求参数可哈希。dict 和 list 不能直接作为参数。")

### 4.3 缓存策略选择

| 机制 | 缓存对象 | 适用场景 | 持久化 |
|------|---------|---------|--------|
| `set_llm_cache(InMemoryCache())` | LLM 调用 | 相同 prompt 重复查询 | 否 |
| `set_llm_cache(RedisCache())` | LLM 调用 | 生产环境多实例共享 | 是 |
| `@lru_cache` | Tool 结果 | 单进程内重复 tool 调用 | 否 |
| `BaseStore` + `InjectedStore()` | 状态/记忆 | 跨会话持久化、历史检索 | 是 |
| 自定义 Redis wrapper | Tool 结果 | 跨会话共享 tool 结果 | 是 |

## 5. Tool 直接作为 Node 使用

**核心洞察**：LangGraph 中，Tool 本身就是一等公民——它实现了 LangChain 的 `Runnable` 接口，可以直接被 `builder.add_node()` 当作普通节点使用。

### 5.1 Tool 直接 add_node（精准业务流程）

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages

# ==========================================
# Tool 直接当 Node 使用
# ==========================================

class OrderState(TypedDict):
    messages: Annotated[list, add_messages]
    order_id: str
    order_info: dict

@tool
def query_order(order_id: str) -> str:
    """查询订单详情。"""
    orders_db = {
        "20250601": {"status": "配送中", "items": ["iPhone 16"], "total": 8999},
        "20250602": {"status": "已签收", "items": ["AirPods"], "total": 1999},
    }
    order = orders_db.get(order_id)
    if not order:
        return f"未找到订单 {order_id}"
    return f"订单 {order_id}: 状态={order['status']}, 商品={', '.join(order['items'])}, 金额=¥{order['total']}"

# 把 tool 直接当节点用
order_builder = StateGraph(OrderState)
order_builder.add_node("query_order", query_order)

# 可以精确编排：先验证身份，再查订单，再生成回复
def verify_user(state: OrderState) -> dict:
    print("[verify_user] 验证用户身份...")
    return {"messages": ["用户身份验证通过"]}

def generate_reply(state: OrderState) -> dict:
    print("[generate_reply] 生成回复...")
    return {"messages": ["回复已生成"]}

order_builder.add_node("verify_user", verify_user)
order_builder.add_node("generate_reply", generate_reply)

order_builder.add_edge(START, "verify_user")
order_builder.add_edge("verify_user", "query_order")
order_builder.add_edge("query_order", "generate_reply")
order_builder.add_edge("generate_reply", END)

order_graph = order_builder.compile()
print("Tool 直接作为 Node 的图编译成功！")
print("流程: START -> verify_user -> query_order -> generate_reply -> END")

### 5.2 ToolNode（LLM 自主决策调用）

当你想让 LLM **自己决定**要不要调、调哪个时，用 `ToolNode` 包装一组 tools。

In [ ]:
from langgraph.prebuilt import ToolNode, tools_condition

# ==========================================
# ToolNode: LLM 自主决策调用工具
# ==========================================

@tool
def get_weather(city: str) -> str:
    """获取指定城市的天气信息"""
    weather_db = {"北京": "晴 25C", "上海": "多云 28C"}
    return weather_db.get(city, f"未找到 {city} 的天气数据")

@tool
def calculate(expression: str) -> str:
    """计算数学表达式"""
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"计算错误: {e}"

tools = [get_weather, calculate]
tool_node = ToolNode(tools)

print(f"ToolNode 包含 {len(tools)} 个工具:")
for t in tools:
    print(f"  - {t.name}: {t.description}")
print()
print("ToolNode 工作原理:")
print("  1. 检查 LLM 输出的 tool_calls 字段")
print("  2. 根据 name 找到对应的 tool 函数")
print("  3. 传入 arguments 执行 tool")
print("  4. 把结果包装成 ToolMessage 返回")

### 5.3 两种方式对比

| 场景 | 方式 | 控制权 |
|------|------|-------|
| 必须按固定顺序执行（先查订单再退款） | Tool 直接 add_node | 开发者精确控制 |
| LLM 自主判断调哪个工具 | ToolNode + tools_condition | LLM 自主决策 |
| 同一个工具在多个图中复用 | Tool 直接 add_node | 定义一次，到处复用 |

> **关键理解**：`ToolNode` 本质上就是 LangGraph 帮你写好的"自动路由节点"——它检查 LLM 的 `tool_calls`，自动分发到对应的 tool。而 Tool 直接 `add_node` 是你在手动编排流程。

## 6. 上下文管理：防止 Tool 结果撑爆窗口

### 6.1 问题：Tool 结果太长怎么办？

用户让查"Python 文档"，Tool 返回了 5000 字的网页内容，塞进 message history → Token 暴涨 → 费用暴涨 → 可能超出模型上下文上限。

### 6.2 策略 1：Truncate（截断）

In [ ]:
# ==========================================
# 策略 1: Tool 结果截断
# ==========================================

@tool
def fetch_url(url: str, max_length: int = 200) -> str:
    """获取网页内容，自动截断过长内容。"""
    # 模拟网页内容
    content = """Python 是一种高级编程语言。它由 Guido van Rossum 于 1991 年创建。
Python 的设计哲学强调代码的可读性和简洁的语法。
它支持多种编程范式，包括面向对象、命令式、函数式和结构化编程。
Python 拥有庞大的标准库和丰富的第三方包生态系统。
它被广泛应用于 Web 开发、数据科学、人工智能、自动化运维等领域。"""
    
    if len(content) > max_length:
        content = content[:max_length] + "\n...[内容已截断]"
    return content

result = fetch_url.invoke({"url": "https://python.org", "max_length": 100})
print(f"截断结果 ({len(result)} 字符):")
print(result)

### 6.3 策略 2：选择性传递

不是把 tool 的原始输出全部塞给 LLM，而是提取关键信息。

In [ ]:
import json

# ==========================================
# 策略 2: 选择性传递
# ==========================================

@tool
def query_weather_selective(city: str) -> str:
    """查询天气，只返回 LLM 需要的关键信息。"""
    # 模拟原始数据（很大）
    raw_data = {
        "city": city,
        "temp": 25,
        "weather": [{"description": "晴"}],
        "humidity": 65,
        "pressure": 1013,
        "visibility": 10000,
        "wind_speed": 3.5,
        "wind_deg": 120,
        "clouds": 20,
        "sunrise": "05:30",
        "sunset": "19:20",
        "coord": {"lat": 39.9, "lon": 116.4},
    }

    # 只提取 LLM 关心的字段
    simplified = {
        "city": raw_data["city"],
        "temperature": raw_data["temp"],
        "condition": raw_data["weather"][0]["description"],
        "humidity": raw_data["humidity"]
    }
    return json.dumps(simplified, ensure_ascii=False)

result = query_weather_selective.invoke({"city": "北京"})
print(f"选择性传递结果:")
print(result)
print(f"\n原始数据约 300 字符，传递约 80 字符，节省 {(1 - 80/300)*100:.0f}%")

### 6.4 策略 3：格式化文本 + State 写入（标准机制）

LangGraph 官方标准做法：
1. **`return` = 精心格式化的自然语言文本**（给 LLM 看，必须进 message）
2. **`state` 写入 = 原始结构化数据**（给下游节点用，不占用 LLM 上下文）

In [ ]:
# ==========================================
# 策略 3: 格式化文本 + State 写入
# ==========================================

class WeatherAgentState(TypedDict):
    messages: Annotated[list, add_messages]
    weather_raw: dict | None   # 原始数据放这里

def weather_node(state: WeatherAgentState) -> dict:
    """查询天气，return 文本给 LLM，原始数据写入 state"""
    city = "北京"  # 从 state 中解析
    raw_data = {"temp": 25, "humidity": 65, "condition": "晴", "wind": "东南风2级"}
    
    # ① return 的只有 LLM 可读的自然语言摘要
    text_summary = (
        f"{city}今天{raw_data['condition']}，"
        f"气温{raw_data['temp']}°C，"
        f"湿度{raw_data['humidity']}%，"
        f"{raw_data['wind']}。"
    )
    
    return {
        "messages": [text_summary],      # 给 LLM 看
        "weather_raw": raw_data,          # 给下游节点用，不进 message
    }

print("核心原则:")
print("  给 LLM 看  -> return 字符串（进入 message history）")
print("  给下游节点 -> state['key'] = value（通过 state 旁路传递）")
print()
print("不要做的事: 把原始 JSON 直接 return 给 LLM")
print("要做的事:   return 永远放自然语言摘要；原始数据通过 state 旁路传递")

### 6.5 五种策略对比

| 策略 | 实现难度 | 信息损失 | 适用场景 |
|------|---------|---------|---------|
| Truncate（截断） | 低 | 高（后半截丢了） | 网页抓取、文档阅读 |
| 选择性传递 | 中 | 低（按需提取） | API 返回结构化数据 |
| Message Trimming | 低 | 中（早期消息丢了） | 长对话场景 |
| Summarization | 高 | 可控（LLM 智能压缩） | 需要保留上下文的复杂对话 |
| 格式化文本 + State 写入 | 中 | 无（原始数据放 state 不发 LLM） | 大结果量 + 下游需原始数据 |

## 7. Prompt Templates 基础

Prompt Template 是 LangChain 中用于**构建可复用、动态化提示词**的工具。

### 7.1 PromptTemplate（字符串模板）

最简单的模板类型，用于 Completion 模式的 LLM。

In [ ]:
from langchain_core.prompts import PromptTemplate

# ==========================================
# PromptTemplate: 字符串模板
# ==========================================

prompt_template = PromptTemplate.from_template(
    "Tell me a joke about {topic}"
)

filled_prompt = prompt_template.invoke({"topic": "cats"})
print(f"模板: {prompt_template.template}")
print(f"填充后: {filled_prompt}")
print()

# 多变量示例
multi_template = PromptTemplate.from_template(
    "You are a {role}. Please explain {concept} in simple terms."
)
multi_result = multi_template.invoke({
    "role": "teacher",
    "concept": "machine learning"
})
print(f"多变量模板: {multi_result}")

### 7.2 ChatPromptTemplate（聊天模板）

用于格式化消息列表的模板，支持角色区分（system/user/assistant）。

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# ==========================================
# ChatPromptTemplate: 聊天模板
# ==========================================

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful {role}"),
    ("user", "Tell me about {topic}")
])

messages = chat_prompt.invoke({
    "role": "weather assistant",
    "topic": "tomorrow's forecast"
})

print("生成的消息列表:")
for msg in messages.messages:
    print(f"  [{msg.type}] {msg.content}")

### 7.3 MessagesPlaceholder（消息占位符）

用于在模板中插入动态消息列表，常用于聊天历史。

In [ ]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

# ==========================================
# MessagesPlaceholder: 消息占位符
# ==========================================

prompt_with_history = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])

chat_history = [
    HumanMessage(content="What's the capital of France?"),
    AIMessage(content="Paris is the capital of France."),
]

messages = prompt_with_history.invoke({
    "chat_history": chat_history,
    "input": "What about Germany?"
})

print("带历史的消息列表:")
for msg in messages.messages:
    print(f"  [{msg.type}] {msg.content}")
print()

# 简化写法
simple_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are helpful"),
    ("placeholder", "{chat_history}"),  # 简写形式
    ("human", "{input}")
])
print("简化写法也支持: (\"placeholder\", \"{variable_name}\")")

### 7.4 FewShotPromptTemplate（少样本提示模板）

用于包含示例的提示模板，实现 Few-shot Learning。

In [ ]:
from langchain_core.prompts import FewShotPromptTemplate

# ==========================================
# FewShotPromptTemplate: 少样本提示
# ==========================================

# 定义示例
examples = [
    {"input": "happy", "output": "sad"},
    {"input": "tall", "output": "short"},
    {"input": "hot", "output": "cold"}
]

# 定义示例模板
example_template = PromptTemplate.from_template(
    "Input: {input}\nOutput: {output}"
)

# 创建 Few-shot 模板
few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_template,
    prefix="Find the antonym:",
    suffix="Input: {word}\nOutput:",
    input_variables=["word"]
)

result = few_shot_prompt.format(word="big")
print("Few-shot 提示:")
print(result)

### 7.5 FewShotChatMessagePromptTemplate（聊天少样本模板）

用于聊天模型的 Few-shot 模板，支持聊天格式的示例。

In [ ]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

# ==========================================
# FewShotChatMessagePromptTemplate
# ==========================================

examples = [
    {"input": "What's 2+2?", "output": "2+2 equals 4."},
    {"input": "What's 5*5?", "output": "5*5 equals 25."}
]

example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "{output}")
])

few_shot_prompt = FewShotChatMessagePromptTemplate(
    examples=examples,
    example_prompt=example_prompt
)

final_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a math tutor"),
    few_shot_prompt,
    ("human", "{input}")
])

messages = final_prompt.invoke({"input": "What's 10/2?"})
print("Few-shot 聊天提示:")
for msg in messages.messages:
    print(f"  [{msg.type}] {msg.content}")

### 7.6 Prompt Template 选择指南

| 模板类型 | 输出格式 | 适用模型 | 使用场景 |
|---------|---------|---------|---------|
| **PromptTemplate** | 字符串 | Completion 模型 | 简单单轮问答 |
| **ChatPromptTemplate** | 消息列表 | Chat 模型 | 多轮对话、角色区分 |
| **FewShotPromptTemplate** | 字符串 + 示例 | Completion 模型 | 需要示例指导 |
| **FewShotChatMessagePromptTemplate** | 消息列表 + 示例 | Chat 模型 | 聊天模型 Few-shot |
| **MessagesPlaceholder** | 动态消息列表 | Chat 模型 | 插入对话历史 |

## 8. 常见误区与注意事项

| 坑 | 现象 | 怎么避免 |
|----|------|---------|
| **Tool docstring 写得太简略** | LLM 不调用或乱传参数 | docstring 就是 LLM 看到的工具说明书，参数、返回值、示例都要写清楚 |
| **Tool 抛异常而不是返回错误** | 整个图中断 | Tool 应该返回错误信息（字符串），让 LLM 决定下一步。抛异常会打断整个流程 |
| **缓存不设 TTL** | 返回过期数据 | 所有缓存必须有 TTL，根据业务新鲜度需求设置 |
| **Tool 返回原始 JSON 给 LLM** | LLM 理解困难，token 浪费 | Tool 必须 return 纯文本摘要；原始 JSON 写入 state，不塞进 message |
| **Tool 结果太长直接塞给 LLM** | Token 暴涨，费用爆炸 | Tool 返回前要做截断或摘要 |

## 9. 速查表 / Cheat Sheet

```python
# 1. @tool 装饰器
@tool
def my_tool(arg: str) -> str:
    """清晰的 docstring = LLM 的说明书"""
    return result

# 2. @tool + args_schema
class Input(BaseModel):
    arg: str = Field(description="参数说明")

@tool(args_schema=Input)
def my_tool(arg: str) -> str:
    return result

# 3. StructuredTool
tool = StructuredTool.from_function(
    func=my_func,
    name="my_tool",
    args_schema=Input,
    handle_tool_error=True
)

# 4. Tool 直接当 Node
builder.add_node("my_tool", my_tool)

# 5. ToolNode
from langgraph.prebuilt import ToolNode, tools_condition
tool_node = ToolNode([tool1, tool2])
builder.add_node("tools", tool_node)
builder.add_conditional_edges("llm", tools_condition)

# 6. return_direct
@tool(return_direct=True)  # 结果直出，不经过 LLM

# 7. 缓存
@tool
@lru_cache(maxsize=128)
def cached_tool(arg: str) -> str:
    return result

# 8. PromptTemplate
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are {role}"),
    MessagesPlaceholder("history"),
    ("human", "{input}")
])
```

**记忆口诀**：

> **"docstring 是说明书，注入替闭包。Tool 当节点能复用，五层兜底要记牢。return 文本 state 存数据。"**

## 10. 课后练习

1. **完善天气 Tool**：用 `args_schema` 定义一个带 `city`、`date`、`include_aqi` 参数的天气查询 tool，要求有完整的 docstring 和参数描述
2. **实现缓存装饰器**：写一个自定义装饰器，给 Tool 添加 5 分钟 TTL 缓存（提示：用 `time.time()` 记录过期时间）
3. **生产级 Tool**：按照五层设计模式，实现一个"查询图书"Tool，包含输入校验、异常处理、结果格式化
4. **上下文管理**：实现一个 fetch_url tool，自动截断超过 500 字符的内容，并提取标题和摘要
5. **Prompt Template 练习**：用 `ChatPromptTemplate` + `MessagesPlaceholder` 创建一个带历史记录的客服机器人提示模板
6. **Tool 复用**：定义一个 `query_order` tool，在两个不同的 StateGraph 中复用它

---

**下一节**: LG-03 预构建 Agent 与流式输出